In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import glob

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snm3C/hicluster'

In [ ]:
rmbkl_paths = glob.glob(f'{projdir}/rmbkl/*.tsv.gz')
len(rmbkl_paths)

### I forgot that I actually need to generate it across a lot more bins to make it smooth
In order to count the number of cis (intra-chromosomal) contacts in each cell and bulk Hi-C data
(28), we divided the contacts into 143 logarithmic bins, the first of which was for contacts that
were separated by less than 1 Kb. Each subsequent bin covered an exponent step of 0.125, using
base 2. Contacts in bins 1-37 were determined to be noisy and were eliminated, leaving bins 38-
141 as the valid bins.

In [ ]:
tmp_path = rmbkl_paths[0]
dists = pd.read_csv(tmp_path, sep='\t', header=None, index_col=None)
# dists = dists.iloc[:,:4]
mask = dists[0] == dists[2]
dists = dists[mask]
final_dists = dists[3] - dists[1]
final_dists.head()

In [ ]:
log2_min = np.log2(2500)        # ≈ 11.29
log2_max = np.log2(250000000) # ≈ 27.89
step = 0.125
log2_bins = np.arange(log2_min, log2_max + step, step)

bin_edges = 2 ** log2_bins  # convert back to basepair units

In [ ]:
[int(x) for x in bin_edges]

In [ ]:
counts_per_bin, _ = np.histogram(final_dists, bins=bin_edges)
contact_probs = counts_per_bin / counts_per_bin.sum()
counts_per_bin[:10], contact_probs[:10]

In [ ]:
%%time

tmp_path = rmbkl_paths[0]
dists = pd.read_csv(tmp_path, sep='\t', header=None, index_col=None)

In [ ]:
# %%time

# tmp_path = rmbkl_paths[0]
# dists = pl.read_csv(tmp_path, separator='\t', has_header=False)

In [ ]:
%%time

def process_path(path):
    dists = pd.read_csv(path, sep='\t', header=None, index_col=None)
#     dists = np.loadtxt(path, delimiter='\t', dtype=int)
    mask = dists[0] == dists[2]
    final_dists = dists[3][mask] - dists[1][mask]
    counts_per_bin, _ = np.histogram(final_dists, bins=bin_edges)
    contact_probs = counts_per_bin / counts_per_bin.sum()
    return contact_probs

results = []

with ProcessPoolExecutor() as executor:
    results = list(tqdm(executor.map(process_path, rmbkl_paths), total=len(rmbkl_paths)))

In [ ]:
contact_dist = np.vstack(results)
contact_dist.shape

In [ ]:
bin_edges.shape

In [ ]:
nuclei = [x.split('/')[-1].split('.')[0] for x in rmbkl_paths]

In [ ]:
contact_df = pd.DataFrame(data=contact_dist, index=nuclei)
# contact_df['donor'] = [x.split('-')[1][1:4] for x in contact_df.index]
# contact_df['donor'].replace({'39D' : 'C39'}, inplace=True)
# contact_df['time'] = [x.split('-')[1][4:] for x in contact_df.index]
# contact_df['time'].replace({'5' : 'D5'}, inplace=True)
# contact_df['time_num'] = [int(x.split('D')[1]) for x in contact_df['time']]
contact_df.shape

In [ ]:
# contact_df.columns = bin_edges[1:]

In [ ]:
contact_df.head()

In [ ]:
contact_df.to_csv(f'{projdir}/csv/distance/merged.csv.gz', sep='\t')

In [ ]:
!date